# TP2 - Informe tecnico

Sistema de Deteccion y Clasificacion de Razas de Perros — IA 5.2 Computer Vision.

## Equipo
- Alumno 1 :
- Alumno 2 :

## 1. Explicacion completa del pipeline

El trabajo implementa un sistema completo de Computer Vision para deteccion y clasificacion de razas de perros. El desarrollo sigue el flujo incremental indicado por la consigna:

```text
Embeddings -> Busqueda por similitud -> Clasificacion supervisada -> Deteccion -> Pipeline completo
```

En la **Etapa 1**, la imagen se representa mediante un embedding visual. Para eso se utiliza un modelo preentrenado como extractor de caracteristicas, se guarda cada embedding en una base vectorial y luego se recuperan las imagenes mas similares a una consulta. La raza se infiere a partir de los vecinos recuperados.

En la **Etapa 2**, el problema pasa a ser de clasificacion supervisada. Se entrenan modelos que reciben una imagen individual de perro y devuelven una distribucion de probabilidad sobre las 70 razas del dataset. Se comparan dos enfoques: transferencia de aprendizaje con ResNet18 fine-tuned y una CNN propia entrenada desde cero.

En la **Etapa 3**, se integra deteccion y clasificacion. Primero, YOLO preentrenado detecta perros en una imagen completa. Luego, cada bounding box se recorta y se envia al clasificador entrenado en la Etapa 2. El resultado final incluye la imagen original con bounding boxes, raza predicha y scores de confianza.

La infraestructura general —aplicacion Gradio, Docker, estructura del proyecto, persistencia, orquestacion general y funciones auxiliares— fue provista por la catedra. El trabajo se concentro en completar las funciones indicadas para cada etapa e integrarlas con esa infraestructura:

- Etapa 1: `extract_embedding(image)`, `search_similar_images(embedding, top_k)` y `predict_breed_from_neighbors(results)`.
- Etapa 2: `train_classifier()` y `evaluate_classifier()`.
- Etapa 3: `detect_dogs(image)` y `classify_detected_dog(crop)`.

En la **Etapa 1**, una vez recuperados los vecinos más similares, la predicción de raza se realiza mediante `predict_breed_from_neighbors`: un voto mayoritario ponderado por score de similitud entre los `top_k` vecinos recuperados. Si el score del vecino más cercano no supera un umbral de similitud configurable (`similarity_threshold`), el sistema retorna "unknown" en lugar de forzar una predicción con baja confianza.

## 2. Dataset

Se utilizo el dataset **70 Dog Breeds Image Dataset**, compuesto por imagenes de perros organizadas por raza. El dataset principal se encuentra dividido en entrenamiento, validacion y prueba:

| Split | Imagenes | Razas |
|---|---:|---:|
| Train | 7946 | 70 |
| Valid | 700 | 70 |
| Test | 700 | 70 |

En el conjunto de entrenamiento se observo un desbalance moderado: las razas tienen entre 65 y 198 imagenes, con un promedio de 113.5 imagenes y un desvio estandar de 24.7. Ademas, 19 de las 70 razas tienen menos de 100 imagenes.

Este desbalance afecta de manera distinta a cada etapa. En la Etapa 1, no se entrena el extractor de embeddings, pero las razas con mas imagenes tienen mayor representacion en la base vectorial. En la Etapa 2, el desbalance puede incidir mas directamente en el entrenamiento supervisado, porque el modelo aprende a partir de la frecuencia y variabilidad disponible por clase.

Tambien se construyo un conjunto independiente de evaluacion externa, con 26 imagenes distribuidas en 5 razas. Estas imagenes fueron descargadas fuera del dataset principal y se usaron para evaluar la generalizacion ante cambios de dominio, como variaciones de iluminacion, fondo, pose, encuadre y calidad de imagen.

Para la Etapa 3 se preparo ademas un conjunto externo especifico con tres escenarios solicitados por la consigna:

| Caso | Archivo | Objetivo |
|---|---|---|
| Un perro | `01_one_dog.png` | Verificar deteccion y clasificacion de una sola instancia |
| Multiples perros | `02_multiple_dogs.jpeg` | Verificar multiples bounding boxes y clasificacion por crop |
| Escena compleja | `03_complex_scene.jpg` | Evaluar deteccion con fondo, objetos o contexto visual no controlado |

## 3. Preprocesamiento

El preprocesamiento se ajusto a las necesidades de cada etapa.

En las Etapas 1 y 2 las imagenes se redimensionan a `224x224`, se convierten a tensor y se normalizan con media y desvio estandar de ImageNet:

```text
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]
```

Esta normalizacion es necesaria porque los modelos ResNet utilizados fueron preentrenados con esa distribucion de entrada. Si se omitiera, el modelo recibiria imagenes con una escala diferente a la esperada y la calidad de los embeddings o de la clasificacion podria degradarse.

Como las imagenes se cargan con OpenCV, inicialmente se encuentran en formato BGR. Antes de procesarlas con modelos de `torchvision`, se convierten a RGB, ya que los modelos preentrenados esperan ese orden de canales.

En la Etapa 1 no se aplico data augmentation. El extractor se usa solo en modo inferencia para generar embeddings, por lo que alterar artificialmente las imagenes al indexarlas podria introducir inconsistencias en la base vectorial.

En la Etapa 2 si se aplico data augmentation durante entrenamiento:

- Horizontal flip.
- Rotacion moderada.
- Variaciones de brillo y contraste.

Estas transformaciones buscan mejorar la generalizacion del modelo frente a cambios razonables de pose, iluminacion y encuadre. En validacion y test no se aplico augmentation, para medir el desempeno sobre imagenes no alteradas artificialmente.

En la Etapa 3 el preprocesamiento se aplica sobre los crops generados por YOLO. Cada recorte se convierte de BGR a RGB, se redimensiona a `224x224`, se normaliza con ImageNet y se envia al clasificador de Etapa 2. Esto asegura que el clasificador reciba entradas consistentes con las usadas durante su entrenamiento.

Se reviso una muestra de 1076 imagenes y no se detectaron archivos corruptos o ilegibles, por lo que no fue necesario aplicar un filtrado adicional.

## 4. Justificacion de los modelos elegidos

### ResNet18 como extractor de embeddings

En la Etapa 1 se utilizo **ResNet18 preentrenada en ImageNet** como extractor de embeddings, eliminando su capa final de clasificacion. Se toma la representacion previa al clasificador, luego del average pooling global, obteniendo un vector de 512 dimensiones.

La decision se justifica porque ResNet18 ya aprendio caracteristicas visuales generales, como bordes, texturas, formas y patrones de objetos. Esto permite comparar imagenes de perros sin entrenar un modelo desde cero. Ademas, ResNet18 tiene un costo computacional menor que arquitecturas mas grandes, lo que facilita la generacion de embeddings y la busqueda por similitud.

### ResNet18 Fine-Tuned

En la Etapa 2 se utilizo **ResNet18 Fine-Tuned** como modelo principal de clasificacion supervisada. Se reemplazo la capa final por una salida de 70 clases, se congelo la mayor parte del backbone y se dejo entrenable el ultimo bloque convolucional (`layer4`) junto con la cabeza de clasificacion.

Este enfoque aprovecha transferencia de aprendizaje: el modelo parte de pesos preentrenados en ImageNet y solo ajusta las capas mas cercanas a la salida para especializarse en razas de perros. Esto mejora el desempeno cuando el dataset no es masivo y contiene clases visualmente similares.

### CNN Custom

Tambien se entreno una **CNN propia** como modelo comparativo. Esta arquitectura contiene bloques convolucionales con Batch Normalization, ReLU y MaxPooling, seguidos de Adaptive Average Pooling, Dropout y una capa lineal final.

La CNN Custom tiene menos parametros y menor tiempo de inferencia que ResNet18, pero se entrena desde cero. Por eso debe aprender desde rasgos basicos hasta patrones especificos de cada raza, lo cual resulta mas dificil con un dataset relativamente acotado y desbalanceado.

### YOLOv8n

En la Etapa 3 se utilizo **YOLOv8n preentrenado** para detectar perros. No se entreno un detector propio porque la consigna pide utilizar YOLO preentrenado y concentrarse en la integracion con el clasificador de razas.

YOLO resuelve la localizacion: detecta donde hay perros en la imagen. El clasificador de Etapa 2 resuelve la identificacion de raza sobre cada crop. Esta separacion reduce el ruido de fondo y permite trabajar con imagenes complejas, multiples perros u objetos adicionales.

## 5. Proceso de entrenamiento e hiperparametros

El entrenamiento supervisado de la Etapa 2 se realizo sobre el split `train`, usando `valid` para monitorear el aprendizaje y guardar el mejor checkpoint segun `valid_loss`.

Los hiperparametros principales fueron:

| Hiperparametro | Valor |
|---|---:|
| Batch size | 32 |
| Epochs maximas | 15 |
| Patience | 4 |
| Learning rate cabeza | 1e-3 |
| Learning rate backbone | 1e-4 |
| Scheduler | StepLR |
| Step size | 5 |
| Gamma | 0.5 |
| Optimizador | Adam |
| Funcion de perdida | CrossEntropyLoss |
| Image size | 224 |

Para **ResNet18 Fine-Tuned** se utilizaron learning rates diferenciados: uno menor para `layer4`, que conserva conocimiento preentrenado, y uno mayor para la nueva capa `fc`, que debe aprender desde cero la clasificacion de las 70 razas.

Para **CNN Custom** se entrenaron todos los parametros desde cero con Adam y el learning rate definido para la cabeza. En ambos casos se registro el historial de `train_loss`, `valid_loss`, `train_acc` y `valid_acc`, lo que permitio graficar curvas de entrenamiento y detectar estabilidad, mejora o posible sobreajuste.

El mejor modelo se guardo como checkpoint `.pth`, incluyendo:

- `model_state_dict`;
- `class_names`;
- `model_name`;
- `history`.

Estos checkpoints luego fueron utilizados por la aplicacion y por la Etapa 3 para clasificar los crops detectados por YOLO.

## 6. Resultados obtenidos

### Etapa 1: busqueda por similitud

La Etapa 1 se evaluo con **NDCG@10**, metrica adecuada porque mide la calidad del ranking recuperado. Para cada imagen de test se recuperaron los diez vecinos mas similares y se asigno relevancia segun coincidiera o no la raza.

| Conjunto | Imagenes | Razas | NDCG@10 |
|---|---:|---:|---:|
| Test interno | 700 | 70 | 0.9584 |
| Externo | 26 | 5 | 0.6705 |

El valor alto en test interno muestra que ResNet18 como extractor de embeddings organiza correctamente gran parte del espacio visual del dataset. La caida en el conjunto externo es esperable por cambios de dominio: imagenes con otros fondos, poses, resoluciones, encuadres o condiciones de iluminacion.

### Etapa 2: clasificacion supervisada

Los resultados finales sobre test fueron:

| Modelo | Accuracy | Precision | Recall | Specificity | F1-score |
|---|---:|---:|---:|---:|---:|
| ResNet18 Fine-Tuned | 0.9586 | 0.9601 | 0.9586 | 0.9994 | 0.9566 |
| CNN Custom | 0.3714 | 0.3709 | 0.3714 | 0.9909 | 0.3424 |

La diferencia entre ambos modelos es significativa. ResNet18 Fine-Tuned alcanza un desempeno alto y equilibrado. En cambio, la CNN Custom obtiene un rendimiento mucho menor, aunque con menor costo computacional.

Comparacion de costo:

| Modelo | Parametros | Tiempo de inferencia |
|---|---:|---:|
| ResNet18 Fine-Tuned | 11.212.422 | 2.97 ms |
| CNN Custom | 407.366 | 0.67 ms |

Evaluacion externa:

| Modelo | Accuracy externa | Precision | Recall | F1-score |
|---|---:|---:|---:|---:|
| ResNet18 Fine-Tuned | 0.5385 | 1.0000 | 0.5333 | 0.6552 |
| CNN Custom | 0.1538 | 0.4000 | 0.1400 | 0.2000 |

La evaluacion externa muestra una caida respecto del test interno. Esto era esperable porque las imagenes externas no pertenecen al mismo dominio que el dataset original.

### Etapa 3: deteccion y clasificacion

La Etapa 3 se probo con tres imagenes externas:

| Caso | Archivo | Objetivo |
|---|---|---|
| Un perro | `01_one_dog.png` | Detectar y clasificar una unica instancia |
| Multiples perros | `02_multiple_dogs.jpeg` | Detectar varias instancias en la misma imagen |
| Escena compleja | `03_complex_scene.jpg` | Evaluar deteccion con fondo y objetos adicionales |

Para cada imagen, YOLO genero bounding boxes de perros. Luego, cada crop fue clasificado con ResNet18 Fine-Tuned. El notebook de Etapa 3 muestra visualmente la imagen original con cajas, raza predicha, score de deteccion y score de clasificacion.

Los scores deben interpretarse separadamente: `det_score` indica la confianza de YOLO en que el objeto detectado es un perro, mientras que `breed_score` indica la confianza del clasificador sobre la raza asignada al crop.

### Clases con mayor dificultad de clasificación

En el conjunto de test interno, las razas con peor recall en el Modelo A (ResNet18 Fine-Tuned) fueron aquellas con mayor similitud visual entre sí. El caso más claro fue **Malinois** (pastor belga), que el modelo confundió frecuentemente con **German Shepherd** (pastor alemán): ambas razas comparten estructura corporal, coloración y proporciones similares, lo que hace que sus embeddings sean cercanos en el espacio de características.

En el Modelo B (CNN Custom), se observaron confusiones similares pero más frecuentes y menos predecibles: además del par Malinois/German Shepherd, aparecieron errores entre **Saint Bernard** y **Basset**, razas que no comparten similitud visual evidente, lo que sugiere que la CNN propia no logró aprender las features discriminativas necesarias para separarlas con el tamaño de dataset disponible.

Estas confusiones son consistentes con la brecha de desempeño entre ambos modelos: ResNet18 Fine-Tuned, al partir de representaciones preentrenadas en ImageNet, dispone de features más ricas y generalizables que la CNN entrenada desde cero.

## 7. Comparacion entre enfoques

### Busqueda por similitud vs clasificacion supervisada

La busqueda por similitud de la Etapa 1 es util porque no requiere entrenar un clasificador de razas. Utiliza embeddings preentrenados y recupera vecinos visualmente cercanos. Esto permite una solucion interpretable: se puede observar que imagenes justifican la prediccion.

Sin embargo, su prediccion depende fuertemente de la calidad y representatividad de la base vectorial. Si una raza tiene pocos ejemplos o si la imagen de consulta esta fuera del dominio del dataset, el ranking puede degradarse.

La clasificacion supervisada de la Etapa 2 aprende directamente una frontera de decision entre las 70 clases. ResNet18 Fine-Tuned obtuvo mejores metricas globales en test interno que la busqueda por similitud como mecanismo de prediccion final, pero requiere entrenamiento, seleccion de hiperparametros y evaluacion de sobreajuste.

### ResNet18 Fine-Tuned vs CNN Custom

ResNet18 Fine-Tuned fue claramente superior en calidad predictiva. Su ventaja principal es la transferencia de aprendizaje: parte de caracteristicas visuales ya aprendidas en ImageNet y solo ajusta las capas finales al dominio de razas de perros.

La CNN Custom es mas liviana y rapida, pero su desempeno fue mucho menor. Al entrenarse desde cero, debe aprender tanto rasgos basicos como patrones especificos de cada raza. Con 70 clases, desbalance moderado y razas visualmente parecidas, esa tarea resulto considerablemente mas dificil.

### YOLO + clasificador

En la Etapa 3, YOLO y ResNet18 cumplen roles distintos. YOLO localiza perros en imagenes completas; ResNet18 clasifica cada crop. Esta separacion es necesaria porque el clasificador fue entrenado con imagenes centradas de perros y no con escenas completas con fondos complejos, personas u otros objetos.

## 8. Problemas encontrados y soluciones implementadas

Durante el desarrollo se identificaron varios puntos criticos:

1. **Diferencia entre imagenes del dataset e imagenes externas.**  
   En los conjuntos externos se observo menor rendimiento, especialmente por cambios de fondo, iluminacion, encuadre y pose. Para abordarlo, se separo la evaluacion interna de la externa y se analizo la caida de desempeno como un problema de generalizacion.

2. **Consistencia de preprocesamiento.**  
   Las imagenes cargadas con OpenCV llegan en BGR, mientras que los modelos de `torchvision` esperan RGB. Se incorporo conversion BGR -> RGB antes de extraer embeddings o clasificar crops.

3. **Integracion entre Etapa 2 y Etapa 3.**  
   La Etapa 3 depende de los checkpoints entrenados en la Etapa 2. Para resolverlo, el pipeline reconstruye la arquitectura correspondiente, carga el `state_dict` y utiliza los `class_names` guardados en el checkpoint.

4. **Diferencia entre deteccion y clasificacion.**  
   YOLO devuelve una confianza de deteccion, mientras que el clasificador devuelve una confianza sobre la raza. Se conservaron ambos scores para no mezclar fuentes distintas de incertidumbre.

5. **Pruebas con escenas complejas.**  
   Para cumplir la consigna, se preparo un conjunto externo especifico con tres escenarios: un perro, multiples perros y escena compleja. Esto permite verificar que el pipeline no funciona solamente con imagenes limpias o centradas.

6. **Ejecucion local y rutas.**  
   Para facilitar la ejecucion en Visual Studio Code, se adapto el notebook de Etapa 3 a rutas locales del repositorio: `models/` para checkpoints y `data/external_eval/eval_etapa_3/` para imagenes externas.

## 9. Modificaciones fuera de las funciones indicadas

La consigna indica que el trabajo debe concentrarse en completar funciones puntuales dentro de la infraestructura provista. En ese sentido, la implementacion principal se mantuvo dentro de las funciones requeridas:

- Etapa 1: `extract_embedding`, `search_similar_images`, `predict_breed_from_neighbors`.
- Etapa 2: `train_classifier`, `evaluate_classifier`.
- Etapa 3: `detect_dogs`, `classify_detected_dog`.

No se modifico la arquitectura general de la aplicacion, la interfaz Gradio, la API ni la configuracion Docker como parte central del desarrollo.

Las modificaciones o ajustes adicionales realizados fueron de integracion y ejecucion:

- incorporacion de imports necesarios para PyTorch, torchvision, PIL, OpenCV y YOLO;
- uso de variables configurables para modelo YOLO, threshold, paths, tamanio de imagen y checkpoints;
- adaptacion del notebook de Etapa 3 para ejecucion local en Visual Studio Code;
- organizacion de imagenes externas para cumplir los tres casos de prueba requeridos por la consigna.

Estas modificaciones no alteran el pipeline provisto por la catedra, sino que permiten ejecutar y documentar correctamente las funciones implementadas.

### Detalle de modificaciones de infraestructura

Más allá de los imports y rutas, las modificaciones de integración incluyeron los siguientes cambios estructurales, todos documentados y justificados en los commits del repositorio:

- **`SimilarityService.__init__`** (`similarity_service.py`): se extendió el constructor para cargar el modelo ResNet18 pre-entrenado (sin capa `fc`), seleccionar el dispositivo de cómputo y definir las transformaciones de preprocesamiento. La carga se realiza una sola vez al instanciar la clase, no en cada llamada a `extract_embedding`, para evitar el costo repetido. No se modificó la firma del constructor ni la interfaz pública de la clase.

- **`PgVectorEmbeddingStore.search()`** (`pgvector_store.py`): se agregó un cast explícito `%s::vector` en la cláusula `ORDER BY` de la consulta SQL. Causa: psycopg serializa la lista de floats como `double precision[]` en lugar de `vector`, y el operador `<=>` de pgvector requiere ambos operandos del mismo tipo. Modificación de una línea, sin alterar la firma ni el comportamiento del método.

- **`config.py`**: se agregaron 7 campos configurables vía variables de entorno (`batch_size`, `max_epochs`, `patience`, `lr_head`, `lr_backbone`, `step_size`, `gamma`), siguiendo el patrón ya existente en el proyecto. Esto evita hardcodear hiperparámetros dentro de `classifier_service.py`.

- **`bootstrap.py`**: se pasó `settings=settings` al construir `ClassifierService`, para que el servicio acceda a los hiperparámetros sin recibirlos como argumentos sueltos.

- **`ClassifierService.__init__`** (`classifier_service.py`): se agregó el parámetro `settings: Settings` con import bajo `TYPE_CHECKING` para evitar un ciclo de importación circular.

- **Helpers privados en `ClassifierService`**: se agregaron `_build_transforms`, `_build_resnet18_finetuned` y `_build_cnn_custom`, reutilizados por `train_classifier` y `evaluate_classifier`. El objetivo es que ambas funciones usen exactamente la misma arquitectura y preprocesamiento, evitando desincronización si se modifica una sin actualizar la otra.

Ninguna de estas modificaciones altera el pipeline provisto por la cátedra ni la interfaz pública de los servicios existentes.